In [ ]:
import os
import csv
import torch
import librosa
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torchvision import models
from datetime import datetime
import io
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. SETUP MODEL & LOAD WEIGHTS
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['CargoShip', 'KaiYuan', 'noise', 'SpeedBoat', 'UUv'] 

resnet_model = models.resnet18(weights=None)
resnet_model.fc = torch.nn.Linear(resnet_model.fc.in_features, 5)
resnet_model = resnet_model.to(device)

try:
    weights_path = "../DATA/models/ResNet18_Transfer.pth"
    resnet_model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
    resnet_model.eval()
    print(" Model Weights Loaded Successfully!")
except Exception as e:
    print(f" Error: {e}")

# 2. INFERENCE FUNCTION
def predict_new_audio(audio_data, model):
    TARGET_SR = 22050
    try:
        y_orig, sr_orig = librosa.load(audio_data, sr=None, mono=False)
        y_mono = librosa.to_mono(y_orig) if y_orig.ndim > 1 else y_orig
        y_resampled = librosa.resample(y_mono, orig_sr=sr_orig, target_sr=TARGET_SR)
            
        S = librosa.feature.melspectrogram(y=y_resampled, sr=TARGET_SR, n_mels=128, fmax=8000)
        S_dB = librosa.power_to_db(S, ref=np.max)
        
        tensor_img = torch.tensor(S_dB).unsqueeze(0).unsqueeze(0).float()
        resized_img = F.interpolate(tensor_img, size=(128, 128), mode='bilinear')
        inputs = resized_img.to(device).repeat(1, 3, 1, 1) 
            
        with torch.no_grad():
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            confidence, preds = torch.max(probs, 1)
            return class_names[preds.item()], confidence.item() * 100
    except Exception as e:
        return None, None

# 3. UI WITH LIVE TABLE DISPLAY
def setup_enhanced_interface():
    log_file = "Model_Testing_History.csv"
    if not os.path.exists(log_file):
        pd.DataFrame(columns=["Timestamp", "File Name", "Predicted Class", "Confidence Score (%)"]).to_csv(log_file, index=False)

    upload_btn = widgets.FileUpload(accept='.wav, .mp3', multiple=False, description='Upload Audio 🎵', button_style='info')
    output_area = widgets.Output()

    def update_display():
        with output_area:
            clear_output(wait=True)
            # Latest 5 results dikhane ke liye CSV read karein
            df = pd.read_csv(log_file)
            if not df.empty:
                print("\n  LATEST TESTING RESULTS ")
                # This will show the last 5 entries in reverse order (latest on top)
                display(df.tail(5)[::-1]) 
            else:
                print("No history found. Upload a file to start!")

    def on_upload_change(change):
        if not upload_btn.value: return
        
        uploaded_file = upload_btn.value[0]
        file_name = uploaded_file['name']
        audio_data = io.BytesIO(uploaded_file['content'])
        
        pred_class, conf_score = predict_new_audio(audio_data, resnet_model)
        
        if pred_class:
            # Save to CSV
            timestamp = datetime.now().strftime("%H:%M:%S")
            new_entry = pd.DataFrame([[timestamp, file_name, pred_class, f"{conf_score:.2f}%"]], 
                                     columns=["Timestamp", "File Name", "Predicted Class", "Confidence Score (%)"])
            new_entry.to_csv(log_file, mode='a', header=False, index=False)
            
        upload_btn.value = () # Reset button
        update_display() # Refresh the table

    upload_btn.observe(on_upload_change, names='value')
    display(widgets.VBox([upload_btn, output_area]))
    update_display() # Initial table show

# RUN
setup_enhanced_interface()

 Model Weights Loaded Successfully!
